# ByT5 Silabisasi Predictor (Colab)

Notebook ini untuk inferensi silabisasi pantun modern menggunakan model seq2seq ByT5 yang sudah dilatih dan disimpan di Google Drive.

Alur inferensi:
- Input baris pantun -> prefiks `'silabisasi: '` -> ByT5 (`AutoModelForSeq2SeqLM`)
- Output teks bersilabel (mis. `me-nyu-ka-i si-fat-mu yang ra-mah`) -> ekstraksi jumlah silabel, silabel akhir, dan skema rima bait.


In [ ]:
!pip -q install transformers sentencepiece accelerate pandas tqdm

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Lokasi folder model ByT5 hasil fine-tuning di Google Drive
MODEL_DIR = '/content/drive/MyDrive/Models/byt5-silabel'  # <- sesuaikan jika berbeda

# Parameter generasi (harus selaras dengan konfigurasi saat training)
TASK_PREFIX = 'silabisasi: '
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
NUM_BEAMS = 4
NO_REPEAT_NGRAM_SIZE = 3
REPETITION_PENALTY = 1.2


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if not os.path.exists(MODEL_DIR):
    raise FileNotFoundError(f'Folder model tidak ditemukan: {MODEL_DIR}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device)
model.generation_config.max_length = None
model.eval()
print(f'[ByT5] Loaded model dari: {MODEL_DIR}')

In [ ]:
import re


def predict_silabel_line(text: str) -> str:
    """Prediksi pemenggalan silabel untuk satu baris teks pantun."""
    cleaned = str(text or '').strip()
    if not cleaned:
        return ''

    prefixed = TASK_PREFIX + cleaned
    inputs = tokenizer(
        prefixed,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            max_length=None,
            num_beams=1,
            # no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            # repetition_penalty=REPETITION_PENALTY,
            # early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def predict_silabel_batch(texts: list[str], batch_size: int = 16) -> list[str]:
    """Prediksi silabisasi secara batch agar lebih cepat untuk evaluasi banyak baris."""
    if not texts:
        return []

    results = []
    for start in range(0, len(texts), batch_size):
        chunk = [TASK_PREFIX + str(t or '').strip() for t in texts[start:start + batch_size]]
        inputs = tokenizer(
            chunk,
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
            padding=True,
            return_tensors='pt',
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=NUM_BEAMS,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                repetition_penalty=REPETITION_PENALTY,
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        results.extend([d.strip() for d in decoded])
    return results


def _parse_syllables_from_segmented(segmented_line: str) -> tuple[list[dict], list[str]]:
    """Memecah string bersilabel (mis. 'pa-gi ke pa-sar') menjadi struktur token & list silabel."""
    token_entries = []
    line_syllables = []

    for raw_word in str(segmented_line or '').strip().split():
        clean_word = re.sub(r'^[^\w-]+|[^\w-]+$', '', raw_word)
        if not clean_word:
            continue
        sylls = [s for s in clean_word.split('-') if s]
        if not sylls:
            continue
        token_entries.append({
            'segmented_token': clean_word,
            'token': ''.join(sylls),
            'syllables': sylls,
            'syllable_count': len(sylls),
        })
        line_syllables.extend(sylls)

    return token_entries, line_syllables


def _extract_rhyme_tail_grapheme(syllable_text: str) -> str:
    """Ekstraksi rima akhir dari silabel grafem (mulai dari inti vokal terakhir)."""
    s = re.sub(r'[^a-zA-Z]', '', str(syllable_text or '').lower())
    if not s:
        return ''

    vowels = set('aiueo')
    if len(s) >= 2 and s[-2:] in {'ai', 'au', 'oi'}:
        return s[-2:]

    for i in range(len(s) - 1, -1, -1):
        if s[i] in vowels:
            return s[i:]
    return s


def _rhyme_key(rhyme_tail: str) -> str:
    t = (rhyme_tail or '').strip()
    if not t:
        return '-'
    return t[-2:] if len(t) >= 2 else t


def infer_rhyme_scheme(line_reports: list[dict]) -> str:
    keys = [_rhyme_key(rep.get('final_rhyme', '')) for rep in line_reports]
    mapping = {}
    next_ord = ord('a')
    scheme = []
    for k in keys:
        if k not in mapping:
            mapping[k] = chr(next_ord)
            next_ord += 1
        scheme.append(mapping[k])
    return '-'.join(scheme)


def analyze_line_byt5(text: str) -> dict:
    segmented = predict_silabel_line(text)
    tokens, line_syllables = _parse_syllables_from_segmented(segmented)
    final_syllable = line_syllables[-1] if line_syllables else ''
    final_rhyme = _extract_rhyme_tail_grapheme(final_syllable)

    return {
        'input_text': text,
        'predicted_silabel': segmented,
        'tokens': tokens,
        'line_syllables': line_syllables,
        'line_syllable_count': len(line_syllables),
        'final_syllable': final_syllable,
        'final_rhyme': final_rhyme,
    }


def analyze_pantun_byt5(text: str) -> dict:
    normalized_text = str(text).replace('\r\n', '\n').replace('\r', '\n').replace('\\n', '\n')
    lines = [ln.strip() for ln in normalized_text.splitlines() if ln.strip()]
    line_reports = []
    for i, line in enumerate(lines, start=1):
        rep = analyze_line_byt5(line)
        rep['line_no'] = i
        line_reports.append(rep)

    rhyme_scheme = infer_rhyme_scheme(line_reports)
    total_syllables = sum(rep.get('line_syllable_count', 0) for rep in line_reports)

    return {
        'raw_text': text,
        'line_count': len(line_reports),
        'lines': line_reports,
        'total_syllables_all_lines': total_syllables,
        'rhyme_scheme': rhyme_scheme,
    }


def build_pantun_compact_output(report: dict) -> dict:
    lines = report.get('lines', [])
    compact = {
        'skema_rima': report.get('rhyme_scheme', ''),
        'rima_akhir_baris_1': '',
        'rima_akhir_baris_2': '',
        'rima_akhir_baris_3': '',
        'rima_akhir_baris_4': '',
        'suku_kata_baris_1': '',
        'suku_kata_baris_2': '',
        'suku_kata_baris_3': '',
        'suku_kata_baris_4': '',
    }
    for i in range(min(4, len(lines))):
        rep = lines[i]
        compact[f'rima_akhir_baris_{i+1}'] = rep.get('final_rhyme', '')
        compact[f'suku_kata_baris_{i+1}'] = rep.get('line_syllable_count', '')
    return compact


def print_pantun_byt5_analysis(report: dict, verbose: bool = True) -> None:
    compact = build_pantun_compact_output(report)
    cols = [
        'skema_rima',
        'rima_akhir_baris_1',
        'rima_akhir_baris_2',
        'rima_akhir_baris_3',
        'rima_akhir_baris_4',
        'suku_kata_baris_1',
        'suku_kata_baris_2',
        'suku_kata_baris_3',
        'suku_kata_baris_4',
    ]
    print('Ringkas CSV:', ','.join(str(compact[c]) for c in cols))

    if not verbose:
        return

    print(f"Jumlah baris terdeteksi : {report['line_count']}")
    print(f"Total silabel (semua)   : {report['total_syllables_all_lines']}")
    print(f"Skema rima              : {report['rhyme_scheme']}")

    for line_rep in report['lines']:
        print('\n' + '=' * 64)
        print(f"BARIS {line_rep['line_no']}: {line_rep['input_text']}")
        print(f"Prediksi ByT5      : {line_rep['predicted_silabel']}")
        print(f"Silabel baris      : {line_rep['line_syllables']} (total={line_rep['line_syllable_count']})")
        print(f"Silabel akhir      : {line_rep['final_syllable']} | Rima akhir: {line_rep['final_rhyme']}")
        for idx, tok in enumerate(line_rep['tokens'], start=1):
            print(f"  [{idx:02d}] {tok['token']} -> {tok['syllables']} (n={tok['syllable_count']})")


In [ ]:
# Contoh uji satu baris
# print('Single line:', predict_silabel_line('menyukai sifatmu yang ramah'))
# print()

# Contoh uji input pantun multi-baris (pisahkan dengan newline "\n")
sample_pantun = """pagipagi bikin ktp\nniatnya ingin beli sate\njika kamu cintai pantai\nmari kita menghargai apapun yang digapai"""

pantun_report = analyze_pantun_byt5(sample_pantun)
print_pantun_byt5_analysis(pantun_report, verbose=True)

In [ ]:
# @title Evaluasi Semua Pantun Gaul (ByT5)
JALANKAN_EVALUASI_GAUL = True  # @param {type:"boolean"}
INPUT_GAUL_CSV_PATH = '/content/drive/MyDrive/pantun_gaul2.csv'  # @param {type:"string"}
OUTPUT_GAUL_EVAL_CSV_PATH = '/content/drive/MyDrive/Scripts/pantun_gaul_byt5_eval.csv'  # @param {type:"string"}

if JALANKAN_EVALUASI_GAUL:
    import pandas as pd

    required_cols = [
        'id',
        'text_pantun',
        'skema_rima',
        'rima_akhir_baris_1',
        'rima_akhir_baris_2',
        'rima_akhir_baris_3',
        'rima_akhir_baris_4',
        'suku_kata_baris_1',
        'suku_kata_baris_2',
        'suku_kata_baris_3',
        'suku_kata_baris_4',
    ]

    df = pd.read_csv(INPUT_GAUL_CSV_PATH)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f'Kolom wajib belum lengkap: {missing}')

    from tqdm.auto import tqdm

    df_out = df[required_cols].copy()
    texts = df_out['text_pantun'].fillna('').astype(str).tolist()

    reports = []
    total = len(texts)
    for idx, t in enumerate(tqdm(texts, desc='Inferensi ByT5', unit='pantun'), start=1):
        reports.append(analyze_pantun_byt5(t))
        if idx % 50 == 0 or idx == total:
            print(f'Proses: pantun ke-{idx} dari {total}')
    pred_rows = [build_pantun_compact_output(rep) for rep in reports]
    pred_df = pd.DataFrame(pred_rows)

    pred_cols = [
        'skema_rima',
        'rima_akhir_baris_1',
        'rima_akhir_baris_2',
        'rima_akhir_baris_3',
        'rima_akhir_baris_4',
        'suku_kata_baris_1',
        'suku_kata_baris_2',
        'suku_kata_baris_3',
        'suku_kata_baris_4',
    ]

    for col in pred_cols:
        df_out[f'pred_{col}'] = pred_df[col].fillna('').astype(str).str.strip()

    df_out.to_csv(OUTPUT_GAUL_EVAL_CSV_PATH, index=False)
    print(f"Selesai! File hasil ByT5 tersimpan di: {OUTPUT_GAUL_EVAL_CSV_PATH}")
    display(df_out.head(10))
else:
    print('Evaluasi pantun gaul OFF. Aktifkan JALANKAN_EVALUASI_GAUL untuk menjalankan.')

